# Stage 7.3B — Official SVM (CICIoMT2024)

**GitHub-ready / run cells individually.** This notebook assumes that the repository includes the repaired `colab/test_stage6_colab.py` committed together with this file. **The notebook does not patch source files in Colab.**

1. **CELL 1–3:** inspect runtime, connect Drive, clone once / check clean Git checkout.
2. **CELL 4–5:** validate `MyDrive/WINDOWS_CPLUS`; run all required Stage 7.3B and legacy adapter tests. `MyDrive/CSV_SPLIT_CPLUS` is optional.
3. **CELL 6–9:** select run, inspect identity, resource evidence, and separately reviewed approval.
4. **CELL 10:** intentionally locked; never run without valid human approval and resource budgets.
5. **CELL 11–12:** inspect completed result without re-evaluating Test.

**Do not use Runtime → Run all.** Test results and dataset validation do **not** authorize SVM training. Do not commit Drive data, model artifacts, authorization secrets, or fake test approval metadata.

In [ ]:
# CELL 1 — Colab runtime and paths (no training)
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys
import psutil

# A fresh clone is created in CELL 3, only when missing.
REPO = Path('/content/RFforNetwork_stage73b')
DRIVE_ROOT = Path('/content/drive/MyDrive/COMPACT-MI-RFE')
LOCAL_RUNS = Path('/content/stage73b_runs')
WINDOWS = Path('/content/drive/MyDrive/WINDOWS_CPLUS')

print(json.dumps({
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'cpu_logical': os.cpu_count(),
    'cpu_physical': psutil.cpu_count(logical=False),
    'ram_total_GiB': round(psutil.virtual_memory().total / 2**30, 2),
    'ram_available_GiB': round(psutil.virtual_memory().available / 2**30, 2),
    'disk_free_GiB': round(shutil.disk_usage('/content').free / 2**30, 2),
}, indent=2))
print('No training started. Run each cell manually; do NOT use Run all.')


In [ ]:
# CELL 2 — Connect Google Drive (interactive; no dataset download)
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Google Drive mount did not expose MyDrive')
    print('Drive mount: PASS')
else:
    print('Drive mount intentionally disabled; dataset/approval cells will be BLOCKED.')


In [ ]:
# CELL 3 — Clone once and verify Git source identity
# Must be run AFTER pushing BOTH notebook + fixed colab/test_stage6_colab.py.
# Existing checkout is NEVER reset, pulled, cleaned, or overwritten.
CHECK_SOURCE_IDENTITY = True
CLONE_IF_MISSING = True
GITHUB_URL = 'https://github.com/kasum00/RFforNetwork.git'

if CHECK_SOURCE_IDENTITY:
    if not (REPO / '.git').is_dir():
        if REPO.exists():
            raise RuntimeError(f'{REPO} exists but is not a Git checkout; inspect manually.')
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f'Checkout missing: {REPO}')
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(REPO)], check=True)

    head = subprocess.run(
        ['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
        text=True, capture_output=True, check=True,
    ).stdout.strip()
    dirty = subprocess.run(
        ['git', '-C', str(REPO), 'status', '--porcelain'],
        text=True, capture_output=True, check=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError('Git checkout has local changes; STOP and inspect. Do not auto-clean.')

    required = [
        'colab/test_stage6_colab.py',
        'colab/svm_run_authorization.py',
        'colab/svm_official_supervisor.py',
        'colab/svm_official_readiness.py',
        'colab/dataset_validator.py',
        'colab/experiment_launcher.py',
        'huyen_code/ciciomt24_training.py',
        'huyen_code/ciciomt24_sliding_window.py',
    ]
    missing = [name for name in required if not (REPO / name).is_file()]
    if missing:
        raise FileNotFoundError('Required repository files missing: ' + ', '.join(missing))

    # The Stage 6 adapter test fix MUST live in Git, not be patched by Colab.
    test_source = (REPO / 'colab/test_stage6_colab.py').read_text(encoding='utf-8')
    if ('unit_fixture_only' not in test_source or
            'test_adapter_persists_refit_before_test_and_restores_source_symbols' not in test_source):
        raise RuntimeError(
            'The fixed Stage 6 adapter test is absent from Git. '
            'Commit colab/test_stage6_colab.py together with this notebook first.'
        )

    for p in (str(REPO), str(REPO / 'huyen_code')):
        if p not in sys.path:
            sys.path.insert(0, p)
    from colab.svm_official_readiness import normalized_source_sha256, source_contract
    print(json.dumps({
        'git_commit': head,
        'checkout_clean': True,
        'stage6_test_fix_committed': True,
        'training_sha256_normalized': normalized_source_sha256(
            REPO / 'huyen_code/ciciomt24_training.py'),
        'sliding_sha256_normalized': normalized_source_sha256(
            REPO / 'huyen_code/ciciomt24_sliding_window.py'),
        'source_contract': source_contract(REPO),
    }, indent=2, default=str))
else:
    print('Source identity check disabled; not ready for official SVM.')


In [ ]:
# CELL 4 — Dataset paths and read-only scientific validation
# No copy, no unzip, no sliding-window regeneration.
CHECK_DATASET_READINESS = True
DRIVE_DATASET_PATH = '/content/drive/MyDrive/WINDOWS_CPLUS'
CSV_SPLIT_DATASET_PATH = '/content/drive/MyDrive/CSV_SPLIT_CPLUS'
REQUIRE_CSV_SPLIT = False  # CSV_SPLIT is optional for Stage 7.3B SVM.

def resolve_one_nested_folder(root: Path, folder_name: str) -> Path:
    return root / folder_name if (root / folder_name).is_dir() else root

if CHECK_DATASET_READINESS:
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Google Drive not mounted; run CELL 2 first.')
    WINDOWS = resolve_one_nested_folder(Path(DRIVE_DATASET_PATH), 'WINDOWS_CPLUS')
    CSV_SPLIT_SOURCE = resolve_one_nested_folder(
        Path(CSV_SPLIT_DATASET_PATH), 'CSV_SPLIT_CPLUS')

    print('WINDOWS_CPLUS:', WINDOWS, 'FOUND:', WINDOWS.is_dir())
    print('CSV_SPLIT_CPLUS:', CSV_SPLIT_SOURCE,
          'FOUND:', CSV_SPLIT_SOURCE.is_dir(), '(optional)')
    if not WINDOWS.is_dir():
        raise FileNotFoundError(f'SVM WINDOWS_CPLUS dataset missing: {WINDOWS}')
    if REQUIRE_CSV_SPLIT and not CSV_SPLIT_SOURCE.is_dir():
        raise FileNotFoundError(f'Required CSV_SPLIT_CPLUS missing: {CSV_SPLIT_SOURCE}')

    # Bounded listing. Do not read any Test feature rows in this cell.
    for name in sorted(p.name for p in WINDOWS.iterdir())[:15]:
        print('  ', name)

    for p in (str(REPO), str(REPO / 'huyen_code')):
        if p not in sys.path:
            sys.path.insert(0, p)
    from colab.dataset_validator import validate_dataset
    DATASET_AUDIT = validate_dataset(WINDOWS, REPO, all_headers=True)
    print(json.dumps(DATASET_AUDIT, indent=2, default=str))
    if DATASET_AUDIT.get('status') not in ('PASS', 'PASS_WITH_WARNINGS'):
        raise RuntimeError('Dataset audit did not pass; STOP before training.')
    print('No estimator fit. No Test feature rows read for evaluation.')
else:
    print('Dataset validation disabled; official SVM must stay BLOCKED.')


In [ ]:
# CELL 5 — Test the committed source (no notebook source patching)
# Includes the repaired Stage 6 adapter test + Stage 7.3B authorization tests.
RUN_STAGE73B_TESTS = True
UNIT_TEST_EVIDENCE = None  # Set to a Git-commit-specific file after successful tests.
if RUN_STAGE73B_TESTS:
    if not (REPO / '.git').is_dir():
        raise FileNotFoundError('Run CELL 3 first; no Git checkout exists.')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Google Drive in CELL 2 first.')
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO), str(REPO / 'huyen_code')])
    modules = [
        'colab.test_stage73b_authorization',
        'colab.test_stage73b_supervisor',
        'colab.test_stage73b_scientific_integrity',
        'colab.test_svm_official_readiness',
        'colab.test_stage6_colab',
        'huyen_code.test_ciciomt24_svm_probe',
        'huyen_code.test_ciciomt24_svm_scaling_probe',
    ]
    print('Running mock/unit tests only; no real SVM fit or official Test evaluation.')
    completed = subprocess.run(
        [sys.executable, '-m', 'unittest', '-v', *modules],
        cwd=REPO, env=env, text=True, capture_output=True,
    )
    print(completed.stdout)
    print(completed.stderr)
    if completed.returncode:
        raise RuntimeError('Unit tests FAILED; no new PASS evidence saved; no training.')

    from colab.dataset_validator import sha256
    git_commit = subprocess.run(
        ['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    UNIT_TEST_EVIDENCE = (
        DRIVE_ROOT / 'environment' / f'stage73b_unit_tests_{git_commit[:12]}.json'
    )
    evidence = {
        'passed': True,
        'modules': modules,
        'returncode': completed.returncode,
        'git_commit': git_commit,
        'training_sha256': sha256(REPO / 'huyen_code/ciciomt24_training.py'),
        'adapter_test_sha256': sha256(REPO / 'colab/test_stage6_colab.py'),
        'full_training_authorized': False,
    }
    UNIT_TEST_EVIDENCE.parent.mkdir(parents=True, exist_ok=True)
    if UNIT_TEST_EVIDENCE.exists():
        existing = json.loads(UNIT_TEST_EVIDENCE.read_text(encoding='utf-8'))
        if existing != evidence:
            raise FileExistsError(
                f'Old/different evidence exists at {UNIT_TEST_EVIDENCE}. '
                'Do not overwrite it silently. Choose a new UNIT_TEST_EVIDENCE path '
                'or archive the old file after manual review.'
            )
        print('Unit evidence already matches this Git commit and tested source; reusing.')
    else:
        UNIT_TEST_EVIDENCE.write_text(json.dumps(evidence, indent=2), encoding='utf-8')
        print('New unit evidence:', UNIT_TEST_EVIDENCE)
    print('CELL 5: PASS. This does NOT authorize official SVM training.')
else:
    print('Unit tests disabled; official SVM remains BLOCKED.')


In [ ]:
# CELL 6 — Select exactly one experiment/config/run (no fit)
EXPERIMENT = 'baseline_svm'      # 'baseline_svm' or 'mi_rfe_svm'
CONFIG = 'FW_512_FS_16'         # FW_256_FS_8, FW_256_FS_16, FW_512_FS_8, FW_512_FS_16
RUN_NAME = 'baseline_svm_FW512_FS16_official01'
APPROVAL = DRIVE_ROOT / 'approvals' / f'{RUN_NAME}.json'
TECHNICAL_EVIDENCE = DRIVE_ROOT / 'technical_evidence' / 'REPLACE_WITH_APPROVED_FILE.json'

if EXPERIMENT not in ('baseline_svm', 'mi_rfe_svm'):
    raise ValueError('Unsupported SVM experiment')
if CONFIG not in ('FW_256_FS_8', 'FW_256_FS_16', 'FW_512_FS_8', 'FW_512_FS_16'):
    raise ValueError('Unsupported sliding window configuration')
print(json.dumps({
    'experiment': EXPERIMENT, 'config': CONFIG, 'run_name': RUN_NAME,
    'approval_exists': APPROVAL.is_file(),
    'technical_evidence_exists': TECHNICAL_EVIDENCE.is_file(),
    'execution_gate': 'BLOCKED by default',
}, indent=2))


In [ ]:
# CELL 7 — Collect run identity in read-only mode (NO approval created)
RUN_READ_ONLY_READINESS = True
if RUN_READ_ONLY_READINESS:
    if not WINDOWS.is_dir():
        raise FileNotFoundError(f'Run CELL 4 first: {WINDOWS}')
    for p in (str(REPO), str(REPO / 'huyen_code')):
        if p not in sys.path:
            sys.path.insert(0, p)
    from colab.experiment_launcher import RunSpec
    from colab.svm_run_authorization import collect_identity
    identity, identity_evidence = collect_identity(
        REPO, WINDOWS, RunSpec(EXPERIMENT, CONFIG, RUN_NAME))
    print(json.dumps({
        'identity': identity,
        'identity_evidence': identity_evidence,
        'status': 'BLOCKED_NO_EFFECTIVE_APPROVAL',
    }, indent=2, default=str))
else:
    print('Identity collection disabled; official SVM BLOCKED.')


In [ ]:
# CELL 8 — Review bounded technical evidence (never auto-approves full run)
REVIEW_RESOURCE_EVIDENCE = True
if REVIEW_RESOURCE_EVIDENCE:
    if not TECHNICAL_EVIDENCE.is_file():
        print('BLOCKED: no approved technical evidence at', TECHNICAL_EVIDENCE)
    else:
        probe = json.loads(TECHNICAL_EVIDENCE.read_text(encoding='utf-8'))
        fields = [
            'status', 'experiment', 'config', 'sizes', 'max_tree_rss_bytes',
            'elapsed_seconds', 'resource_gate', 'full_training_authorized',
        ]
        print(json.dumps({key: probe.get(key) for key in fields}, indent=2))
        if (probe.get('experiment'), probe.get('config')) != (EXPERIMENT, CONFIG):
            print('BLOCKED: evidence not equivalent to chosen experiment/config.')
        if probe.get('full_training_authorized') is not False:
            print('BLOCKED: technical probe must never authorize training itself.')
    print('This cell cannot grant full-training approval.')
else:
    print('Technical evidence review disabled; official SVM BLOCKED.')


In [ ]:
# CELL 9 — Validate explicit run-specific approval (default OFF)
REVIEW_EFFECTIVE_APPROVAL = False
AUTHORIZATION_CONFIRMATION = ''  # Exact phrase supplied by the human-approved workflow.

if REVIEW_EFFECTIVE_APPROVAL:
    if UNIT_TEST_EVIDENCE is None:
        raise RuntimeError('Run CELL 5 tests successfully first.')
    for path in (UNIT_TEST_EVIDENCE, APPROVAL, TECHNICAL_EVIDENCE):
        if not path.is_file():
            raise FileNotFoundError(f'Authorization input missing: {path}')
    if not AUTHORIZATION_CONFIRMATION.startswith('AUTHORIZE STAGE7.3B '):
        raise PermissionError('A full, exact Stage 7.3B confirmation phrase is required.')
    for p in (str(REPO), str(REPO / 'huyen_code')):
        if p not in sys.path:
            sys.path.insert(0, p)
    from colab.experiment_launcher import RunSpec
    from colab.svm_run_authorization import authorize_run
    unit = json.loads(UNIT_TEST_EVIDENCE.read_text(encoding='utf-8'))
    if unit.get('passed') is not True:
        raise RuntimeError('Unit test evidence is not PASS.')
    AUTH_CONTEXT = authorize_run(
        REPO, WINDOWS, LOCAL_RUNS, DRIVE_ROOT,
        RunSpec(EXPERIMENT, CONFIG, RUN_NAME), APPROVAL,
        TECHNICAL_EVIDENCE, AUTHORIZATION_CONFIRMATION,
        unit_test_evidence=unit,
    )
    print(json.dumps(AUTH_CONTEXT.public_record(), indent=2, default=str))
else:
    print('BLOCKED: no effective run-specific approval reviewed.')


In [ ]:
# CELL 10 — OFFICIAL SVM EXECUTION: deliberately LOCKED by default
# Enable ONLY after source/dataset/resource audits and run-specific human approval.
START_OFFICIAL_SVM = False
I_ACCEPT_SINGLE_FINAL_TEST_EVALUATION = False

if not START_OFFICIAL_SVM or not I_ACCEPT_SINGLE_FINAL_TEST_EVALUATION:
    raise RuntimeError('Official SVM is BLOCKED; no training started. DO NOT Run all.')
if not REVIEW_EFFECTIVE_APPROVAL:
    raise PermissionError('CELL 9 approval review has not been enabled and validated.')
if 'AUTH_CONTEXT' not in globals():
    raise PermissionError('Missing validated authorization context from CELL 9.')
if not AUTHORIZATION_CONFIRMATION.startswith('AUTHORIZE STAGE7.3B '):
    raise PermissionError('Exact run-specific authorization phrase missing.')
if UNIT_TEST_EVIDENCE is None:
    raise RuntimeError('CELL 5 unit test evidence is missing.')
for path in (APPROVAL, TECHNICAL_EVIDENCE, UNIT_TEST_EVIDENCE):
    if not path.is_file():
        raise FileNotFoundError(path)

cmd = [
    sys.executable, '-m', 'colab.svm_official_supervisor',
    '--repo', str(REPO), '--windows', str(WINDOWS),
    '--local-runs', str(LOCAL_RUNS), '--drive-root', str(DRIVE_ROOT),
    '--experiment', EXPERIMENT, '--config', CONFIG, '--run-name', RUN_NAME,
    '--approval', str(APPROVAL), '--technical-evidence', str(TECHNICAL_EVIDENCE),
    '--unit-test-evidence', str(UNIT_TEST_EVIDENCE),
    '--authorization-confirmation', AUTHORIZATION_CONFIRMATION,
]
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([str(REPO), str(REPO / 'huyen_code')])
subprocess.run(cmd, cwd=REPO, env=env, check=True)


In [ ]:
# CELL 11 — Inspect final artifact metadata only; never re-evaluate Test
VERIFY_COMPLETED_ARTIFACT = False
if VERIFY_COMPLETED_ARTIFACT:
    result_dir = DRIVE_ROOT / 'experiment_results' / RUN_NAME
    result_path = result_dir / 'result.json'
    manifest_path = result_dir / 'artifact_manifest.json'
    for path in (result_path, manifest_path):
        if not path.is_file():
            raise FileNotFoundError(path)
    result = json.loads(result_path.read_text(encoding='utf-8'))
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if result.get('test_evaluation_calls') != 1:
        raise RuntimeError('Expected exactly one final Test evaluation.')
    required = {
        'result.json', 'model.joblib', 'final_scaler.joblib',
        'authorization.json', 'hyperparameters.json', 'feature_schema.json',
    }
    absent = required - set(manifest.get('files', {}))
    if absent:
        raise RuntimeError('Artifact manifest missing: ' + ', '.join(sorted(absent)))
    print(json.dumps({
        'status': result.get('status'),
        'artifact_files': len(manifest['files']),
        'test_evaluation_calls': result['test_evaluation_calls'],
        'test_re_evaluation_performed_here': False,
    }, indent=2))
else:
    print('Post-run artifact verification disabled until a legitimate completed run.')


In [ ]:
# CELL 12 — Read result JSON summary only (NO Test prediction)
READ_RESULT_SUMMARY = False
if READ_RESULT_SUMMARY:
    path = DRIVE_ROOT / 'experiment_results' / RUN_NAME / 'result.json'
    if not path.is_file():
        raise FileNotFoundError(path)
    result = json.loads(path.read_text(encoding='utf-8'))
    keys = [
        'status', 'experiment', 'config', 'rows_by_split',
        'label_counts_by_split', 'metrics', 'elapsed_seconds',
        'peak_rss_bytes_sampled', 'support_vectors', 'fit_warnings',
        'label_origin', 'capture_session_independence', 'test_evaluation_calls',
    ]
    print(json.dumps({key: result.get(key) for key in keys}, indent=2, default=str))
else:
    print('Summary disabled until authorized training is completed.')
